CPVD

In [1]:
import cpevd
import matplotlib.pyplot as plt
GT_MASK_PATH = r"E:\CPVD\1.TE_Mask.bmp"
PRED_MASK_PATH = r"E:\CPVD\1.TE_MaskPred.bmp"
# Load the user masks
gt_mask = cpevd.load_mask(GT_MASK_PATH)
pred_mask = cpevd.load_mask(PRED_MASK_PATH)

# Calculate metrics and details
results = cpevd.compute_all_metrics(gt_mask, pred_mask)
cpevd_results = cpevd.complex_phase_vector_displacement_similarity_detailed(gt_mask, pred_mask)

print("\n" + "="*50)
print("PERFORMANCE EVALUATION")
print("="*50)
print(f" CPEVD Similarity Index : {cpevd_results['cpevd']:.6f}")
print(f" Dice Coefficient      : {results['dice']:.4f}")
print(f" Jaccard Index (IoU)   : {results['jaccard']:.4f}")
print(f" Accuracy              : {results['accuracy']:.4f}")
print(f" Precision             : {results['precision']:.4f}")
print(f" Sensitivity (Recall)  : {results['sensitivity']:.4f}")
print(f" Specificity           : {results['specificity']:.4f}")
print(f" Hausdorff Distance    : {results['hausdorff']:.2f} pixels")
print("="*50)


PERFORMANCE EVALUATION
 CPEVD Similarity Index : 0.000000
 Dice Coefficient      : 0.0000
 Jaccard Index (IoU)   : 0.0000
 Accuracy              : 0.8831
 Precision             : 0.0000
 Sensitivity (Recall)  : 0.0000
 Specificity           : 1.0000
 Hausdorff Distance    : 466.00 pixels


In [6]:
import os
import glob
from collections import defaultdict
import cv2
import cpevd

# Define folder paths
GT_DIR = r"E:\CPVD\As metrics\Dataset\ICM Original"
PRED_DIR = r"E:\CPVD\As metrics\Dataset\ICM Pred"

# Target shape (width, height)
TARGET_SIZE = (256, 256)

# Supported image extensions
IMAGE_EXTENSIONS = ('*.bmp', '*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff')

# Collect all files from GT directory
gt_files = []
for ext in IMAGE_EXTENSIONS:
    gt_files.extend(glob.glob(os.path.join(GT_DIR, ext)))

# Storage for metric totals
metrics_sum = defaultdict(float)
valid_pairs_count = 0

print("Processing image pairs...")

for gt_path in gt_files:
    filename = os.path.basename(gt_path)
    pred_path = os.path.join(PRED_DIR, filename)

    # Check if corresponding predicted mask exists
    if not os.path.exists(pred_path):
        print(f"Warning: Corresponding prediction not found for {filename}. Skipping.")
        continue

    try:
        # Load the user masks
        gt_mask = cpevd.load_mask(gt_path)
        pred_mask = cpevd.load_mask(pred_path)

        # Resize both masks to 256x256
        # INTER_NEAREST is recommended for binary/label masks to preserve values
        gt_mask = cv2.resize(gt_mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
        pred_mask = cv2.resize(pred_mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)

        # Calculate metrics and details
        results = cpevd.compute_all_metrics(gt_mask, pred_mask)
        cpevd_results = cpevd.complex_phase_vector_displacement_similarity_detailed(gt_mask, pred_mask)

        # Accumulate metrics
        metrics_sum['cpevd'] += cpevd_results['cpevd']
        metrics_sum['dice'] += results['dice']
        metrics_sum['jaccard'] += results['jaccard']
        metrics_sum['accuracy'] += results['accuracy']
        metrics_sum['precision'] += results['precision']
        metrics_sum['sensitivity'] += results['sensitivity']
        metrics_sum['specificity'] += results['specificity']
        metrics_sum['hausdorff'] += results['hausdorff']

        valid_pairs_count += 1

    except Exception as e:
        print(f"Error processing {filename}: {e}")

# Calculate averages and print summary
if valid_pairs_count > 0:
    avg = {key: val / valid_pairs_count for key, val in metrics_sum.items()}

    print("\n" + "="*50)
    print(f"AVERAGE PERFORMANCE EVALUATION ({valid_pairs_count} Images")
    print("="*50)
    print(f" CPEVD Similarity Index : {avg['cpevd']:.6f}")
    print(f" Dice Coefficient       : {avg['dice']:.4f}")
    print(f" Jaccard Index (IoU)    : {avg['jaccard']:.4f}")
    print(f" Accuracy               : {avg['accuracy']:.4f}")
    print(f" Precision              : {avg['precision']:.4f}")
    print(f" Sensitivity (Recall)   : {avg['sensitivity']:.4f}")
    print(f" Specificity            : {avg['specificity']:.4f}")
    print(f" Hausdorff Distance     : {avg['hausdorff']:.2f} pixels")
    print("="*50)
else:
    print("No matching image pairs found to process.")

Processing image pairs...

AVERAGE PERFORMANCE EVALUATION (249 Images
 CPEVD Similarity Index : 0.585710
 Dice Coefficient       : 0.7466
 Jaccard Index (IoU)    : 0.6562
 Accuracy               : 0.9741
 Precision              : 0.9071
 Sensitivity (Recall)   : 0.6879
 Specificity            : 0.9966
 Hausdorff Distance     : 35.29 pixels


In [4]:
import numpy as np
print(np.__version__)

2.2.6
